# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)


In [8]:
# load data from Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
from pathlib import Path

# base data path
data_path = Path("/content/drive/MyDrive/Colab Notebooks/COMP90042NLP/COMP90042_Assignment3/data")

# all file paths
data_path_train_claims = data_path / "train-claims.json"
data_path_evidence = data_path / "evidence.json"
data_path_dev_claims = data_path / "dev-claims.json"
data_path_dev_baseline_claims = data_path / "dev-claims-baseline.json"
data_path_test_claims = data_path / "test-claims-unlabelled.json"


In [10]:
import json

with open(data_path_train_claims) as file:
    train_claims = json.load(file)
# train_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores claim_text, claim_label and evidences.

with open(data_path_evidence) as file:
    evidence = json.load(file)
# evidence is a dictionary with evidence_id as key and the actual text
# as value.

with open(data_path_dev_claims) as file:
    dev_claims = json.load(file)
# dev_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores claim_text, claim_label and evidences.

with open(data_path_dev_baseline_claims) as file:
    dev_baseline_claims = json.load(file)
# dev_baseline_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores claim_text, claim_label and evidences.

with open(data_path_test_claims) as file:
    test_claims = json.load(file)
# test_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores just claim_text.

In [11]:
# first_3 = dict(list(test_claims.items())[:3])

# print(json.dumps(first_3, indent=4, ensure_ascii=False))

In [12]:
import re

def clean_text(text):
    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text


def process_data(data, labelled=True):
    processed_data = []
    for claim_id, value in data.items():
        claim_text = clean_text(value["claim_text"])
        if labelled:
            label = value["claim_label"]
            evidence_ids = value["evidences"]
            evidence_texts = [clean_text(evidence[eid]) for eid in value["evidences"]]
            combined_input = f"Claim: {claim_text} \n\nEvidence:\n" + \
                                "\n".join([f"- {e}" for e in evidence_texts])

        if labelled:
            processed_data.append({
                "claim_id": claim_id,
                "claim_text": claim_text,
                "label": label,
                "evidence_ids": evidence_ids,
                "evidence_texts": evidence_texts,
                "gold_input": combined_input})
        else:
            processed_data.append({
                "claim_id": claim_id,
                "claim_text": claim_text,})
    return processed_data


processed_train_data = process_data(train_claims)
processed_dev_data = process_data(dev_claims)
# processed_train_data, processed_dev_data and processed_dev_baseline_data are lists
# with different claims stored as a dictionary with claim_id, claim_text, label, evidence_ids,
# evidence_text and gold_input. gold_input is a input format for classification.

processed_test_data = process_data(test_claims, labelled=False)
# processed_test_data is a list with different claims stored as a dictionary with
# just claim_id and claim_text.

In [13]:
all_evidence_ids = list(evidence.keys())
all_evidence_texts = [clean_text(evidence[eid]) for eid in all_evidence_ids]
# all_evidence_ids and all_evidence_texts are coresponding lists of all the evidence passages.

In [14]:
print(f"all_evidence_ids.length = {len(all_evidence_ids)}")
print(f"len(processed_train_data) = {len(processed_train_data)}")
print(f"len(processed_dev_data) = {len(processed_dev_data)}")
print(f"len(processed_test_data) = {len(processed_test_data)}")
print(all_evidence_texts[0:3])

all_evidence_ids.length = 1208827
len(processed_train_data) = 1228
len(processed_dev_data) = 154
len(processed_test_data) = 153
['john bennet lawes, english entrepreneur and agricultural scientist', 'lindberg began his professional career at the age of 16, eventually moving to new york city in 1977.', "``boston (ladies of cambridge)'' by vampire weekend"]


# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)


## 2.1 Evidence Retrieval
Given claim_text, output evidence_ids

In [23]:
# setting device on GPU if available, else CPU
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



In [24]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build TF-IDF index over all evidence passages
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
evidence_tfidf = vectorizer.fit_transform(all_evidence_texts)

In [25]:
import copy
# Input: claim_text
# Return: top-k evidence_ids
def retrieve_topk_evidence_ids(claim_text, top_k=5):
    # Vectorize the claim with the same fitted TF-IDF vocabulary.
    claim_clean = clean_text(claim_text)
    claim_vec = vectorizer.transform([claim_clean])

    # Rank evidence passages by cosine similarity in TF-IDF space.
    scores = cosine_similarity(claim_vec, evidence_tfidf).flatten()
    top_indices = scores.argsort()[-top_k:][::-1]
    return [all_evidence_ids[i] for i in top_indices]

def build_retrieval_predictions(claims_dict, top_k=5):
    # Build output
    predictions = copy.deepcopy(claims_dict)

    for claim_id, item in claims_dict.items():
        # Replace evidences with retrieved top-k evidences.
        predictions[claim_id]["evidences"] = retrieve_topk_evidence_ids(
            item["claim_text"],
            top_k=top_k,
        )
        print(f"claim_id= {claim_id} claim_text= {item["claim_text"]}")
        print(predictions[claim_id]["evidences"])
    return predictions


# Example: retrieve top-5 evidence on dev set.
dev_retrieval_top5 = build_retrieval_predictions(dev_claims, top_k=5)

# Quick sanity check for one sample.
sample_claim_id = next(iter(dev_retrieval_top5))
print("Sample claim_id:", sample_claim_id)
print("Retrieved top-5 evidence IDs:", dev_retrieval_top5[sample_claim_id]["evidences"])
print("Number of dev claims:", len(dev_retrieval_top5))

claim_id= claim-752 claim_text= [South Australia] has the most expensive electricity in the world.
['evidence-572512', 'evidence-67732', 'evidence-147175', 'evidence-65625', 'evidence-509525']
claim_id= claim-375 claim_text= when 3 per cent of total annual global emissions of carbon dioxide are from humans and Australia prod­uces 1.3 per cent of this 3 per cent, then no amount of emissions reductio­n here will have any effect on global climate.
['evidence-1140012', 'evidence-833215', 'evidence-234964', 'evidence-364039', 'evidence-1154181']
claim_id= claim-1266 claim_text= This means that the world is now 1C warmer than it was in pre-industrial times
['evidence-694262', 'evidence-911037', 'evidence-38305', 'evidence-755581', 'evidence-240270']
claim_id= claim-871 claim_text= “As it happens, Zika may also be a good model of the second worrying effect — disease mutation.
['evidence-382824', 'evidence-1178347', 'evidence-1067605', 'evidence-395835', 'evidence-597838']
claim_id= claim-2164

## 2.2 Classification

In [1]:
# Authenticate to Hugging Face
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
# Install the requirements in Google Colab
!pip install -q -U transformers accelerate datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [5]:
# BERT fine-tuning for 4-way claim classification (claim + gold evidence text).
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "google-bert/bert-base-uncased"
LABEL_LIST = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]
label2id = {name: i for i, name in enumerate(LABEL_LIST)}
id2label = {i: name for name, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
from datasets import Dataset
from transformers import DataCollatorWithPadding


def claims_to_dataset(processed_rows):
    return Dataset.from_dict({
        "text": [row["gold_input"] for row in processed_rows],
        "labels": [label2id[row["label"]] for row in processed_rows],
    })


train_hf = claims_to_dataset(processed_train_data)
dev_hf = claims_to_dataset(processed_dev_data)


def tokenize_batch(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)


train_tok = train_hf.map(tokenize_batch, batched=True, remove_columns=["text"])
dev_tok = dev_hf.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/1228 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

In [17]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

accuracy_metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)


CLASSIFIER_OUTPUT_DIR = data_path.parent / "bert_classifier_ckpt"
CLASSIFIER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(CLASSIFIER_OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.998839,1.063005,0.532468
2,0.812462,0.923207,0.610390
3,0.725106,0.848812,0.668831


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=462, training_loss=0.8937050656322792, metrics={'train_runtime': 220.0088, 'train_samples_per_second': 16.745, 'train_steps_per_second': 2.1, 'total_flos': 498380343750432.0, 'train_loss': 0.8937050656322792, 'epoch': 3.0})

In [19]:
import torch
def format_claim_evidence_input(claim_text, evidence_ids):
    """Same layout as training `gold_input`, using arbitrary retrieved evidence ids."""
    texts = []
    for eid in evidence_ids:
        if eid in evidence:
            texts.append(clean_text(evidence[eid]))
    if not texts:
        texts = ["[no evidence text]"]
    body = "\n".join(f"- {t}" for t in texts)
    return f"Claim: {clean_text(claim_text)} \n\nEvidence:\n{body}"


def predict_labels_for_texts(texts, batch_size=16):
    model.eval()
    device = next(model.parameters()).device
    out = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i : i + batch_size]
        enc = tokenizer(
            chunk,
            truncation=True,
            max_length=512,
            padding=True,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        for j in logits.argmax(dim=-1).cpu().tolist():
            out.append(id2label[j])
    return out


# Teacher-forcing sanity check (gold evidence passages)
_dev_texts = [row["gold_input"] for row in processed_dev_data]
_dev_pred = predict_labels_for_texts(_dev_texts)
print(
    "Dev accuracy (gold evidence inputs):",
    sum(a == b["label"] for a, b in zip(_dev_pred, processed_dev_data)) / len(processed_dev_data),
)

Dev accuracy (gold evidence inputs): 0.6688311688311688


# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [27]:
import numpy as np


def _evidence_fscore(pred_evidences, gold_evidences):
    """Per-claim retrieval F-score (same definition as eval.py)."""
    evidence_correct = 0
    evidence_recall = 0.0
    evidence_precision = 0.0
    evidence_fscore = 0.0
    if isinstance(pred_evidences, list) and len(pred_evidences) > 0:
        pred_set = set(pred_evidences)
        for gr_ev in gold_evidences:
            if gr_ev in pred_set:
                evidence_correct += 1
        if evidence_correct > 0:
            evidence_recall = float(evidence_correct) / len(gold_evidences)
            evidence_precision = float(evidence_correct) / len(pred_evidences)
            evidence_fscore = (2 * evidence_precision * evidence_recall) / (
                evidence_precision + evidence_recall
            )
    return evidence_fscore


def evaluate_predictions(predictions, groundtruth, verbose=False):
    """Full metrics matching `eval.py`: mean F, classification accuracy, harmonic mean."""
    f_list, acc_list = [], []

    for claim_id, claim in sorted(groundtruth.items()):
        if claim_id not in predictions:
            continue
        pred = predictions[claim_id]
        if "claim_label" not in pred or "evidences" not in pred:
            continue

        instance_correct = 1.0 if pred["claim_label"] == claim["claim_label"] else 0.0
        pred_evs = pred["evidences"]
        evidence_fscore = _evidence_fscore(pred_evs, claim["evidences"])

        if verbose:
            pred_set = set(pred_evs) if isinstance(pred_evs, list) else set()
            evidence_correct = sum(1 for gr_ev in claim["evidences"] if gr_ev in pred_set)
            evidence_recall = (
                float(evidence_correct) / len(claim["evidences"])
                if evidence_correct > 0
                else 0.0
            )
            evidence_precision = (
                float(evidence_correct) / len(pred_evs)
                if evidence_correct > 0 and isinstance(pred_evs, list)
                else 0.0
            )
            print("groundtruth =", claim)
            print("predictions =", pred)
            print("instance accuracy =", instance_correct)
            print("evidence recall =", evidence_recall)
            print("evidence precision =", evidence_precision)
            print("evidence fscore =", evidence_fscore, "\n")

        acc_list.append(instance_correct)
        f_list.append(evidence_fscore)

    mean_f = float(np.mean(f_list if len(f_list) > 0 else [0.0]))
    mean_acc = float(np.mean(acc_list if len(acc_list) > 0 else [0.0]))
    if mean_f == 0.0 and mean_acc == 0.0:
        hmean = 0.0
    else:
        hmean = (2 * mean_f * mean_acc) / (mean_f + mean_acc)

    print("Evidence Retrieval F-score (F)    =", mean_f)
    print("Claim Classification Accuracy (A) =", mean_acc)
    print("Harmonic Mean of F and A          =", hmean)
    return {"mean_f": mean_f, "accuracy": mean_acc, "harmonic_mean": hmean}


def evaluate_retrieval_fscore(predictions, groundtruth, verbose=False):
    """Retrieval F only; ignores `claim_label` in predictions."""
    f_scores = []

    for claim_id, claim in sorted(groundtruth.items()):
        if claim_id not in predictions or "evidences" not in predictions[claim_id]:
            continue
        pred_evs = predictions[claim_id]["evidences"]
        evidence_fscore = _evidence_fscore(pred_evs, claim["evidences"])

        if verbose:
            print("claim_id =", claim_id)
            print("gold evidences =", claim["evidences"])
            print("pred evidences =", pred_evs)
            print("evidence fscore =", evidence_fscore, "\n")

        f_scores.append(evidence_fscore)

    mean_f = float(np.mean(f_scores if len(f_scores) > 0 else [0.0]))
    print("Evidence Retrieval F-score (F)    =", mean_f)
    return mean_f


def evaluate_classification_accuracy(predictions, groundtruth):
    """Accuracy over claim labels only (matches eval.py aggregation over instances)."""
    correct, total = 0, 0
    for claim_id, claim in sorted(groundtruth.items()):
        if claim_id not in predictions or "claim_label" not in predictions[claim_id]:
            continue
        total += 1
        if predictions[claim_id]["claim_label"] == claim["claim_label"]:
            correct += 1
    acc = float(correct) / total if total else 0.0
    print("Claim Classification Accuracy (A) =", acc)
    return acc

In [28]:
import numpy as np


def official_eval_like_script(predictions, groundtruth, verbose=False):
    """Same metrics as eval.py. Returns (F, A, harmonic_mean)."""
    f_scores, acc = [], []
    for claim_id, claim in sorted(groundtruth.items()):
        if claim_id not in predictions:
            continue
        pred = predictions[claim_id]
        if "claim_label" not in pred or "evidences" not in pred:
            continue

        instance_correct = 1.0 if pred["claim_label"] == claim["claim_label"] else 0.0

        evidence_correct = 0
        evidence_recall = 0.0
        evidence_precision = 0.0
        evidence_fscore = 0.0
        if isinstance(pred["evidences"], list) and len(pred["evidences"]) > 0:
            pred_set = set(pred["evidences"])
            for gr_ev in claim["evidences"]:
                if gr_ev in pred_set:
                    evidence_correct += 1
            if evidence_correct > 0:
                evidence_recall = float(evidence_correct) / len(claim["evidences"])
                evidence_precision = float(evidence_correct) / len(pred["evidences"])
                evidence_fscore = (2 * evidence_precision * evidence_recall) / (
                    evidence_precision + evidence_recall
                )

        if verbose:
            print("groundtruth =", claim)
            print("predictions =", pred)
            print("instance accuracy =", instance_correct)
            print("evidence recall =", evidence_recall)
            print("evidence precision =", evidence_precision)
            print("evidence fscore =", evidence_fscore, "\n\n")

        acc.append(instance_correct)
        f_scores.append(evidence_fscore)

    mean_f = float(np.mean(f_scores if len(f_scores) > 0 else [0.0]))
    mean_acc = float(np.mean(acc if len(acc) > 0 else [0.0]))
    if mean_f == 0.0 and mean_acc == 0.0:
        hmean = 0.0
    else:
        hmean = (2 * mean_f * mean_acc) / (mean_f + mean_acc)

    return mean_f, mean_acc, hmean


def build_pipeline_predictions(retrieval_dict, batch_size=16):
    """Attach BERT claim_label using retrieved evidences."""
    claim_ids = sorted(retrieval_dict.keys())
    texts = [
        format_claim_evidence_input(
            retrieval_dict[cid]["claim_text"],
            retrieval_dict[cid]["evidences"],
        )
        for cid in claim_ids
    ]
    labels = predict_labels_for_texts(texts, batch_size=batch_size)
    return {
        cid: {
            "claim_label": lab,
            "evidences": list(retrieval_dict[cid]["evidences"]),
        }
        for cid, lab in zip(claim_ids, labels)
    }


for _need in (
    "build_retrieval_predictions",
    "format_claim_evidence_input",
    "predict_labels_for_texts",
    "dev_claims",
):
    if _need not in globals():
        raise RuntimeError(
            f"Missing `{_need}`: run section 1 (data), 2.1 (retrieval), and 2.2 (classification) first."
        )

try:
    dev_retrieval_top5
except NameError:
    dev_retrieval_top5 = build_retrieval_predictions(dev_claims, top_k=5, verbose=False)

dev_pipeline_predictions = build_pipeline_predictions(dev_retrieval_top5)

F, A, H = official_eval_like_script(
    dev_pipeline_predictions, dev_claims, verbose=False
)
print("Evidence Retrieval F-score (F)    =", F)
print("Claim Classification Accuracy (A) =", A)
print("Harmonic Mean of F and A          =", H)


Evidence Retrieval F-score (F)    = 0.062208822923108656
Claim Classification Accuracy (A) = 0.37662337662337664
Harmonic Mean of F and A          = 0.10678020878723164


## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*